# Deferred Technique Ablations

This notebook collects techniques that are not part of the main baseline yet: arithmetic coding, direct-access indexed streams, sigma-Huffman, and MM-RePair. The default cells run a tiny deterministic CPU probe. MM-RePair is prepared behind an explicit flag because it depends on an external `irepair` binary.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLES_DIR = REPO_ROOT / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "outputs"

from pathlib import Path

import pandas as pd
import torch

from neuralzip.experiment_utils import (
    collect_field,
    ensure_irepair,
    exponent_budget_bits,
    global_huffman_exponent_bits,
    run_irepair,
)
from neuralzip.neuralzip_model import NeuralZipModel

try:
    from IPython.display import display
except ImportError:
    display = print

In [2]:
torch.manual_seed(67)
probe_model = NeuralZipModel([
    ("probe.block0.weight", torch.randn(128, 64, dtype=torch.float32) * 0.25),
    ("probe.block1.weight", torch.randn(64, 64, dtype=torch.float32) * 2.0),
    ("probe.norm.bias", torch.randn(64, dtype=torch.float32) * 0.01),
])

symbols = collect_field(probe_model, "exponent", max_symbols=0)
raw_exponent_bits = exponent_budget_bits(probe_model, symbols.size)
layer_exponents = probe_model.collect_exponents()

deferred_rows = [
    {
        "technique": "global_huffman",
        "scope": "exponent stream",
        "bits": int(global_huffman_exponent_bits(probe_model, layer_exponents)),
    },
    {
        "technique": "arithmetic_order1",
        "scope": "exponent stream estimate",
        "bits": int(probe_model.compute_zipnn_bits(symbols, method="arith")),
    },
    {
        "technique": "sigma_huffman_order1",
        "scope": "exponent stream estimate",
        "bits": int(probe_model.compute_zipnn_bits(symbols, method="sigma")),
    },
    {
        "technique": "direct_access_indexed_stream",
        "scope": "full indexed stream including sign and mantissa",
        "bits": int(probe_model.build_global_exponent_index(block_size=256, ngram=3)),
    },
]

deferred_table = pd.DataFrame(deferred_rows)
deferred_table["ratio_vs_raw_exponent_bits"] = deferred_table["bits"] / raw_exponent_bits
display(deferred_table)

[Exp bitstream indexed (block=256, n=3)]     0.33 Mbits |     0.04 MB | 83.70% of original


,technique,scope,bits,ratio_vs_raw_exponent_bits
0,global_huffman,exponent stream,39856,0.403335
1,arithmetic_order1,exponent stream estimate,43751,0.442752
2,sigma_huffman_order1,exponent stream estimate,34875,0.352929
3,direct_access_indexed_stream,full indexed stream including sign and mantissa,330817,3.347808


In [3]:
RUN_MM_REPAIR = False
MM_REPAIR_DIR = Path("/tmp/mm-repair")
MM_REPAIR_MEMORY_MB = 4096

if RUN_MM_REPAIR:
    irepair = ensure_irepair(MM_REPAIR_DIR, build=False)
    repair_result = run_irepair(irepair, symbols, MM_REPAIR_MEMORY_MB)
    mm_repair_table = pd.DataFrame([
        {
            "technique": "mm_repair",
            "scope": "exponent int32 stream",
            "estimated_bits": int(repair_result["estimated_bytes"] * 8),
            "file_bits": int((repair_result["rule_file_bytes"] + repair_result["sequence_file_bytes"]) * 8),
            "raw_file_bits": int(repair_result["raw_bytes"] * 8),
            "rules": repair_result["rules"],
            "final_sequence_length": repair_result["final_length"],
        }
    ])
    display(mm_repair_table)
else:
    display(pd.DataFrame([
        {
            "technique": "mm_repair",
            "status": "prepared_not_executed",
            "required_binary": str(MM_REPAIR_DIR / "brepair" / "irepair"),
        }
    ]))

,technique,status,required_binary
0,mm_repair,prepared_not_executed,/tmp/mm-repair/brepair/irepair
